**This notebook is an exercise in the [SQL](https://www.kaggle.com/learn/intro-to-sql) course.  You can reference the tutorial at [this link](https://www.kaggle.com/dansbecker/select-from-where).**

---


# Introduction

Try writing some **SELECT** statements of your own to explore a large dataset of air pollution measurements.

Run the cell below to set up the feedback system.* 

In [14]:
# Set up feedback system
from learntools.core import binder
binder.bind(globals())
from learntools.sql.ex2 import *
print("Setup Complete")

Setup Complete


The code cell below fetches the `global_air_quality` table from the `openaq` dataset.  We also preview the first five rows of the table.

# 📊 Global Air Quality Framework & Relational Data Integrity
**Project Overview:** This project leverages Google BigQuery and SQL to programmatically interface with large-scale, multi-variable population databases from OpenAQ. The objective is to build clean data-filtering pipelines, isolate global standardized units of measurement (`ppm`), and analyze baseline air quality metrics across international reporting boundaries to evaluate data integrity.


In [15]:
from google.cloud import bigquery

client = bigquery.Client()

dataset_ref = client.dataset("openaq", project="bigquery-public-data")

table_ref = dataset_ref.table("global_air_quality")

table = client.get_table(table_ref)

client.list_rows(table, max_results=5).to_dataframe()

Using Kaggle's public dataset BigQuery integration.


,location,city,country,pollutant,value,timestamp,unit,source_name,latitude,longitude,averaged_over_in_hours,location_geom
0,"Borówiec, ul. Drapałka",Borówiec,PL,bc,0.85217,2022-04-28 07:00:00+00:00,µg/m³,GIOS,1.0,52.276794,17.074114,POINT(52.276794 1)
1,"Kraków, ul. Bulwarowa",Kraków,PL,bc,0.91284,2022-04-27 23:00:00+00:00,µg/m³,GIOS,1.0,50.069308,20.053492,POINT(50.069308 1)
2,"Płock, ul. Reja",Płock,PL,bc,1.41000,2022-03-30 04:00:00+00:00,µg/m³,GIOS,1.0,52.550938,19.709791,POINT(52.550938 1)
3,"Elbląg, ul. Bażyńskiego",Elbląg,PL,bc,0.33607,2022-05-03 13:00:00+00:00,µg/m³,GIOS,1.0,54.167847,19.410942,POINT(54.167847 1)
4,"Piastów, ul. Pułaskiego",Piastów,PL,bc,0.51000,2022-05-11 05:00:00+00:00,µg/m³,GIOS,1.0,52.191728,20.837489,POINT(52.191728 1)


# Exercises

### 1) Units of measurement

Which countries have reported pollution levels in units of "ppm"?  In the code cell below, set `first_query` to an SQL query that pulls the appropriate entries from the `country` column.

In case it's useful to see an example query, here's some code from the tutorial:

```
query = """
        SELECT city
        FROM `bigquery-public-data.openaq.global_air_quality`
        WHERE country = 'US'
        """
```

### 🌐 Phase 1: Isolating International Reporting Metrics (ppm)


For the solution, uncomment the line below.

In [16]:
query = """
    SELECT DISTINCT country
    FROM `bigquery-public-data.openaq.global_air_quality`
    WHERE unit = 'ppm'
"""

query_job = client.query(query)
results = query_job.result()

print("Countries with pollution levels in ppm:")
for row in results:
    print(row.country)

Countries with pollution levels in ppm:
IL
PE
AU
CH
GT
MM
BH
RW
CA
ZA
QA
CW
RO
CO
EC
BR
TH
US
BM
TW
AR
MX
AE
NP
GB
UZ
IN
CL


### 2) High air quality

Which pollution levels were reported to be exactly 0?  
- Set `zero_pollution_query` to select **all columns** of the rows where the `value` column is 0.
- Set `zero_pollution_results` to a pandas DataFrame containing the query results.

For the solution, uncomment the line below.

### 🔬 Phase 2: Establishing Baseline Clean-Air Controls (Value = 0)

In [17]:
from google.cloud import bigquery

client = bigquery.Client()

zero_pollution_query = """
    SELECT *
    FROM `bigquery-public-data.openaq.global_air_quality`
    WHERE value = 0
"""

safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)
query_job = client.query(zero_pollution_query, job_config=safe_config)

zero_pollution_results = query_job.to_dataframe()

print(zero_pollution_results.head())

Using Kaggle's public dataset BigQuery integration.


/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


                       location          city country pollutant  value  \
0    Żary, ul. Szymanowskiego 8          Żary      PL        bc    0.0   
1       Starachowice, ul. Złota  Starachowice      PL        bc    0.0   
2    Żary, ul. Szymanowskiego 8          Żary      PL        bc    0.0   
3  Koszalin, ul. Armii Krajowej      Koszalin      PL        bc    0.0   
4       Starachowice, ul. Złota  Starachowice      PL        bc    0.0   

                  timestamp   unit source_name  latitude  longitude  \
0 2022-05-18 15:00:00+00:00  µg/m³        GIOS       1.0  51.642656   
1 2022-05-07 11:00:00+00:00  µg/m³        GIOS       1.0  51.050611   
2 2022-05-04 16:00:00+00:00  µg/m³        GIOS       1.0  51.642656   
3 2022-05-17 14:00:00+00:00  µg/m³        GIOS       1.0  54.193986   
4 2022-05-15 14:00:00+00:00  µg/m³        GIOS       1.0  51.050611   

   averaged_over_in_hours       location_geom  
0               15.127808  POINT(51.642656 1)  
1               21.084175  POINT

That query wasn't too complicated, and it got the data you want. But these **SELECT** queries don't organizing data in a way that answers the most interesting questions. For that, we'll need the **GROUP BY** command. 

If you know how to use [`groupby()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html) in pandas, this is similar. But BigQuery works quickly with far larger datasets.

Fortunately, that's next.

# Keep going
**[GROUP BY](https://www.kaggle.com/dansbecker/group-by-having-count)** clauses and their extensions give you the power to pull interesting statistics out of data, rather than receiving it in just its raw format.

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/intro-to-sql/discussion) to chat with other learners.*